# NovaSuite — B2B SaaS Analytics
## Churn Analysis

---

**Objective:** Identify which customer segments churn most, when, and why — using only customer attributes, usage behavior, and stated churn reasons.

---

### Sections
1. Overall Churn Rate
2. Churn by Industry
3. Churn by Plan
4. Churn by Company Size
5. Segment Analysis (Industry × Plan, Size × Plan)
6. Churn Reason Analysis
7. Usage vs Churn
8. Feature Adoption vs Churn
9. High-Risk Customer Profiles
10. Churn Findings Summary

In [ ]:
import pandas as pd
import numpy as np

# ── Load raw tables ──────────────────────────────────────────────────────────
customers = pd.read_csv('customers.csv')
plans     = pd.read_csv('plans.csv')
churn     = pd.read_csv('churn.csv')
usage     = pd.read_csv('usage.csv')
revenue   = pd.read_csv('revenue.csv')

# ── Build master: one row per customer ───────────────────────────────────────
master = customers.copy()

# Join plan details
master = master.merge(plans, on='plan_id', how='left')

# Join churn info (customers not in churn table get NaN → not churned)
master = master.merge(
    churn[['customer_id', 'churn_date', 'reason']],
    on='customer_id',
    how='left'
)

# Average usage metrics per customer (across all their months)
usage_avg = usage.groupby('customer_id')[[
    'active_users', 'sessions', 'login_count', 'feature_usage'
]].mean().reset_index()

usage_avg.columns = [
    'customer_id', 'avg_active_users', 'avg_sessions',
    'avg_login_count', 'avg_feature_usage'
]

master = master.merge(usage_avg, on='customer_id', how='left')

# Most-used feature per customer (the feature that appeared most months)
top_feat = (
    usage.groupby('customer_id')['top_feature']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
top_feat.columns = ['customer_id', 'top_feature']

master = master.merge(top_feat, on='customer_id', how='left')

# ── Churn flag ───────────────────────────────────────────────────────────────
master['is_churned'] = master['churn_date'].notnull()

print('master shape:', master.shape)
print('Columns:', list(master.columns))

---
## 1. Overall Churn Rate

**Business Question:** How severe is churn overall?

Before diving into segments, we need a baseline. The overall churn rate tells us how big the problem is across all 500 customers.

In [ ]:
# ── Count total and churned customers ────────────────────────────────────────
total_customers = master['customer_id'].nunique()
churned_customers = master['is_churned'].sum()      # True = 1, False = 0
active_customers = total_customers - churned_customers

churn_rate = round((churned_customers / total_customers) * 100, 2)
retention_rate = round(100 - churn_rate, 2)

# ── Display as a summary table ────────────────────────────────────────────────
summary = pd.DataFrame({
    'Metric': [
        'Total Customers',
        'Active Customers',
        'Churned Customers',
        'Overall Churn Rate (%)',
        'Overall Retention Rate (%)'
    ],
    'Value': [
        total_customers,
        active_customers,
        churned_customers,
        f'{churn_rate}%',
        f'{retention_rate}%'
    ]
})

print('━' * 40)
print('   OVERALL CHURN SUMMARY')
print('━' * 40)
print(summary.to_string(index=False))
print('━' * 40)

### 📌 Interpretation

- **36 out of 500 customers** have churned — a **7.2% overall churn rate**.
- The KPI target is **< 5% churn**, so NovaSuite is currently **2.2 percentage points above target**.
- At this rate, NovaSuite loses roughly 1 customer every 2–3 weeks.
- This baseline gives us the benchmark to compare against every segment below.

---
## 2. Churn by Industry

**Business Question:** Which industries are losing customers fastest?

NovaSuite serves 8 industries. If churn is concentrated in specific verticals, it may signal a product-market fit problem in those segments.

In [ ]:
# ── Group by industry ─────────────────────────────────────────────────────────
industry_churn = master.groupby('industry').agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

# Calculate churn rate and sort descending
industry_churn['churn_rate_%'] = round(
    (industry_churn['churned'] / industry_churn['total_customers']) * 100, 2
)

industry_churn = industry_churn.sort_values('churn_rate_%', ascending=False)
industry_churn = industry_churn.reset_index(drop=True)

# Add a column flagging above-average churn
avg_churn = churn_rate   # 7.2% from Section 1
industry_churn['vs_avg'] = industry_churn['churn_rate_%'].apply(
    lambda x: '⚠ Above avg' if x > avg_churn else '✓ Below avg'
)

print('━' * 60)
print('   CHURN RATE BY INDUSTRY  (sorted: highest → lowest)')
print('━' * 60)
print(industry_churn.to_string(index=False))
print('━' * 60)
print(f'   Baseline overall churn rate: {avg_churn}%')
print('━' * 60)

### 📌 Interpretation

- **EdTech has a 15.0% churn rate** — more than **double the 7.2% baseline**. With 60 customers and 9 churned, it is the single most at-risk industry.
- **SaaS/Tech (7.27%)** and **Healthcare (7.25%)** are just above the baseline, signaling mild elevated risk.
- **E-Commerce (4.69%)** and **Retail (4.92%)** are the most stable industries — both already below the 5% target.
- The EdTech anomaly is large enough to warrant a dedicated deep-dive in the segment analysis below.

> **Key flag:** EdTech accounts for 25% of all churns (9 of 36) but only 12% of all customers (60 of 500). It is severely over-represented in churn.

---
## 3. Churn by Plan

**Business Question:** Which subscription plans have the highest churn?

If certain plans are losing customers faster, it could indicate pricing pressure, feature mismatch, or plan-to-customer-size misalignment.

In [ ]:
# ── Group by plan ─────────────────────────────────────────────────────────────
plan_churn = master.groupby('plan_name').agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

# Add plan price for context
plan_price_map = {'Basic': 29, 'Pro': 79, 'Business': 149, 'Enterprise': 299}
plan_churn['monthly_price_$'] = plan_churn['plan_name'].map(plan_price_map)

# Churn rate
plan_churn['churn_rate_%'] = round(
    (plan_churn['churned'] / plan_churn['total_customers']) * 100, 2
)

plan_churn = plan_churn.sort_values('churn_rate_%', ascending=False).reset_index(drop=True)

# Revenue at risk: churned customers × monthly price
plan_churn['revenue_at_risk_$'] = plan_churn['churned'] * plan_churn['monthly_price_$']

print('━' * 72)
print('   CHURN RATE BY PLAN  (sorted: highest → lowest)')
print('━' * 72)
print(plan_churn.to_string(index=False))
print('━' * 72)

# Total monthly revenue at risk
total_at_risk = plan_churn['revenue_at_risk_$'].sum()
print(f'\n   Total monthly revenue at risk from churned customers: ${total_at_risk:,}')

### 📌 Interpretation

- **Pro plan has the highest churn rate at 10.17%** — significantly above the 7.2% baseline. It also accounts for the **most churned customers (18)**, making it the biggest churn problem by volume.
- **Basic plan (4.58%)** and **Enterprise plan (3.57%)** both perform below the 7.2% average — Basic is affordable enough that customers stay, and Enterprise customers have deep platform investment.
- **Business plan (7.89%)** is slightly above baseline, suggesting some customers are overshooting their needs at this tier.
- The **Pro plan** is the most commercially concerning: high customer count × high churn rate × mid-range price. It needs investigation into *who* is on this plan and whether they belong there.

---
## 4. Churn by Company Size

**Business Question:** Do Small, Medium, Large, or Enterprise customers churn more?

Company size often correlates with budget stability, IT maturity, and willingness to invest in a platform long-term.

In [ ]:
# ── Group by company size ─────────────────────────────────────────────────────
size_churn = master.groupby('company_size').agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

size_churn['churn_rate_%'] = round(
    (size_churn['churned'] / size_churn['total_customers']) * 100, 2
)

size_churn['pct_of_total_churn'] = round(
    (size_churn['churned'] / size_churn['churned'].sum()) * 100, 2
)

size_churn = size_churn.sort_values('churn_rate_%', ascending=False).reset_index(drop=True)

print('━' * 70)
print('   CHURN RATE BY COMPANY SIZE  (sorted: highest → lowest)')
print('━' * 70)
print(size_churn.to_string(index=False))
print('━' * 70)

# What % of the customer base is Small?
small_pct = round(
    master[master['company_size'] == 'Small'].shape[0] / total_customers * 100, 1
)
print(f'\n   Small customers make up {small_pct}% of all customers')

### 📌 Interpretation

- **Enterprise-sized companies have the highest churn rate at 8.16%** — counterintuitive, as enterprise customers are typically stickier. This may reflect pricing sensitivity or a mismatch between the product's feature depth and enterprise expectations.
- **Small companies (7.45%)** are the second-highest risk group and represent the **largest customer segment (37.6% of all customers)**. Their sheer volume amplifies the business impact of even a small rate increase.
- **Large companies (6.12%)** are the most stable size segment, sitting below the 7.2% baseline.
- Churn is relatively spread across sizes — no single size group is dramatically dominant, which points to plan-level mismatches rather than a size-specific problem alone.

---
## 5. Segment Analysis

**Business Question:** Which specific customer segments have the worst churn?

Looking at two dimensions at once (e.g., Industry *and* Plan) reveals combinations that single-dimension analysis misses. A segment might look fine on one axis but be failing on both.

In [ ]:
# ── 5A. Churn by Industry × Plan ─────────────────────────────────────────────
seg_industry_plan = master.groupby(['industry', 'plan_name']).agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

seg_industry_plan['churn_rate_%'] = round(
    (seg_industry_plan['churned'] / seg_industry_plan['total_customers']) * 100, 2
)

# Keep only segments with at least 1 churn, sort by churn rate
seg_industry_plan = seg_industry_plan[
    seg_industry_plan['churned'] > 0
].sort_values('churn_rate_%', ascending=False).reset_index(drop=True)

print('━' * 64)
print('   CHURN: INDUSTRY × PLAN  (segments with ≥ 1 churn, sorted)')
print('━' * 64)
print(seg_industry_plan.to_string(index=False))
print('━' * 64)

In [ ]:
# ── 5B. Churn by Company Size × Plan ─────────────────────────────────────────
seg_size_plan = master.groupby(['company_size', 'plan_name']).agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

seg_size_plan['churn_rate_%'] = round(
    (seg_size_plan['churned'] / seg_size_plan['total_customers']) * 100, 2
)

seg_size_plan = seg_size_plan[
    seg_size_plan['churned'] > 0
].sort_values('churn_rate_%', ascending=False).reset_index(drop=True)

print('━' * 62)
print('   CHURN: COMPANY SIZE × PLAN  (segments with ≥ 1 churn)')
print('━' * 62)
print(seg_size_plan.to_string(index=False))
print('━' * 62)

### 📌 Interpretation

**Industry × Plan findings:**
- **EdTech + Business plan (23.5% churn)** is the single worst-performing segment. Nearly 1 in 4 EdTech customers on the Business plan has churned.
- **EdTech + Basic (16.7%)** and **EdTech + Pro (11.8%)** both also rank high — EdTech churns across *all* plan tiers, not just one.
- **SaaS/Tech + Pro (15.8%)** is a surprising entry — a tech-savvy segment on the mid-tier plan, possibly underserved on features.
- **Healthcare + Pro (13.8%)** and **Manufacturing + Pro (12.0%)** both cluster around the Pro plan, reinforcing that Pro has the most at-risk customer mix.

**Company Size × Plan findings:**
- **Large + Pro (16.7%)** is the worst size-plan combination. Large companies on the Pro plan are significantly over-churning — they may have outgrown Pro without upgrading.
- **Enterprise + Business (13.6%)** shows Enterprise-sized customers on a plan below Enterprise tier — a likely mismatch.
- **Small + Pro (10.96%)** is high-volume: 73 customers, 8 churned — small companies may be over-committed to a plan beyond their budget or usage needs.

---
## 6. Churn Reason Analysis

**Business Question:** Why are customers leaving?

The `reason` column captures the stated exit reason from churned customers. Understanding the distribution of reasons is the first step toward addressing them.

In [ ]:
# ── Overall reason distribution ───────────────────────────────────────────────
reason_counts = master['reason'].value_counts().reset_index()
reason_counts.columns = ['reason', 'count']
reason_counts['pct_of_churns'] = round(
    (reason_counts['count'] / reason_counts['count'].sum()) * 100, 2
)
reason_counts['cumulative_pct'] = reason_counts['pct_of_churns'].cumsum().round(2)

print('━' * 58)
print('   CHURN REASONS  (sorted by frequency)')
print('━' * 58)
print(reason_counts.to_string(index=False))
print('━' * 58)
print(f"   Total churned customers: {reason_counts['count'].sum()}")
print('━' * 58)

In [ ]:
# ── Reasons broken down by industry ──────────────────────────────────────────
# Filter to churned customers only
churned_only = master[master['is_churned'] == True].copy()

reason_by_industry = churned_only.groupby(['industry', 'reason']).agg(
    count = ('customer_id', 'count')
).reset_index()

reason_by_industry = reason_by_industry.sort_values(
    ['industry', 'count'], ascending=[True, False]
).reset_index(drop=True)

print('━' * 52)
print('   CHURN REASONS BY INDUSTRY')
print('━' * 52)
print(reason_by_industry.to_string(index=False))
print('━' * 52)

### 📌 Interpretation

- **Budget Cut (33.3%)** is the most common churn reason — 12 out of 36 customers left due to spending constraints, not product dissatisfaction.
- **Too Expensive (25.0%)** compounds the budget theme. Combined, **58.3% of churn is price/budget-driven** — the majority of exits are financial, not product-related.
- **Lack of Features (19.4%)** is the leading product-related reason. These are customers who felt the platform didn't meet their needs — often Small or EdTech customers.
- **Switched to Competitor (11.1%)** indicates competitive loss, though relatively contained at 4 customers.
- **Poor Support and Product Complexity** each account for 5.6% — small in total but important signals about the customer experience layer.

> **Key insight:** The top two reasons (Budget Cut + Too Expensive) suggest pricing sensitivity is the dominant driver. This makes the Basic/Pro plan pricing question especially critical for any retention strategy.

---
## 7. Usage vs Churn

**Business Question:** Do churned customers show lower engagement?

If churned customers were already disengaged *before* they left, usage data could serve as an early warning signal. We compare average engagement metrics between churned and active customers.

In [ ]:
# ── Compare average usage: churned vs active ──────────────────────────────────
usage_comparison = master.groupby('is_churned').agg(
    customers        = ('customer_id', 'count'),
    avg_active_users = ('avg_active_users', 'mean'),
    avg_sessions     = ('avg_sessions', 'mean'),
    avg_login_count  = ('avg_login_count', 'mean'),
    avg_feature_usage= ('avg_feature_usage', 'mean')
).round(2).reset_index()

# Make the label readable
usage_comparison['customer_group'] = usage_comparison['is_churned'].map(
    {False: 'Active (not churned)', True: 'Churned'}
)

usage_comparison = usage_comparison[[
    'customer_group', 'customers',
    'avg_active_users', 'avg_sessions', 'avg_login_count', 'avg_feature_usage'
]]

print('━' * 80)
print('   AVERAGE USAGE METRICS: CHURNED vs ACTIVE CUSTOMERS')
print('━' * 80)
print(usage_comparison.to_string(index=False))
print('━' * 80)

In [ ]:
# ── Calculate how much lower churned customers' usage is (% drop) ─────────────
active_row  = usage_comparison[usage_comparison['customer_group'] == 'Active (not churned)'].iloc[0]
churned_row = usage_comparison[usage_comparison['customer_group'] == 'Churned'].iloc[0]

metrics = ['avg_active_users', 'avg_sessions', 'avg_login_count', 'avg_feature_usage']

gap_rows = []
for m in metrics:
    active_val  = active_row[m]
    churned_val = churned_row[m]
    pct_drop = round(((active_val - churned_val) / active_val) * 100, 1)
    gap_rows.append({
        'metric'        : m,
        'active_avg'    : active_val,
        'churned_avg'   : churned_val,
        'churned_is_lower_by_%': pct_drop
    })

gap_df = pd.DataFrame(gap_rows)

print('━' * 62)
print('   HOW MUCH LOWER IS CHURNED CUSTOMER USAGE?')
print('━' * 62)
print(gap_df.to_string(index=False))
print('━' * 62)

### 📌 Interpretation

The usage gap between churned and active customers is substantial across every metric:

| Metric | Active Avg | Churned Avg | Drop |
|--------|-----------|-------------|------|
| Active Users | 23.77 | 14.47 | **-39.1%** |
| Sessions | 139.74 | 65.14 | **-53.4%** |
| Login Count | 316.46 | 148.92 | **-53.0%** |
| Feature Usage Score | 4.18 | 3.39 | **-18.9%** |

- Churned customers had **53% fewer sessions** and **53% fewer logins** on average compared to customers who stayed.
- This is strong evidence that **usage decline precedes churn** — disengagement is a detectable early signal.
- A customer who drops to roughly half their normal session volume may already be evaluating alternatives or preparing to leave.
- The **feature usage score drop (-19%)** is smaller than sessions, suggesting these customers were still using *some* features, but less frequently overall.

---
## 8. Feature Adoption vs Churn

**Business Question:** Are some features associated with lower churn?

If customers who use certain features churn less, those features may represent high "stickiness" — deep product integration that makes customers less likely to leave.

In [ ]:
# ── Churn rate by most-used feature ──────────────────────────────────────────
feature_churn = master.groupby('top_feature').agg(
    total_customers = ('customer_id', 'count'),
    churned         = ('is_churned', 'sum')
).reset_index()

feature_churn['churn_rate_%'] = round(
    (feature_churn['churned'] / feature_churn['total_customers']) * 100, 2
)

feature_churn = feature_churn.sort_values('churn_rate_%').reset_index(drop=True)

# Flag sticky vs at-risk features
feature_churn['signal'] = feature_churn['churn_rate_%'].apply(
    lambda x: '🔒 Sticky' if x < 7.2 else '⚠ At-risk'
)

print('━' * 66)
print('   CHURN RATE BY TOP FEATURE  (sorted: lowest churn first)')
print('━' * 66)
print(feature_churn.to_string(index=False))
print('━' * 66)
print(f'   Baseline churn rate: {churn_rate}%')
print('━' * 66)

### 📌 Interpretation

- **Integrations (3.7%)** and **Dashboard (4.1%)** are the stickiest features — customers whose primary feature is Integrations or Dashboard churn at half the baseline rate. Integrations in particular often create technical dependencies that make switching costly.
- **Reports (5.6%)** is also below average, suggesting data-heavy users are moderately retained.
- **Data Export (19.2%)** is the highest-churn feature — customers primarily using the platform to export data may not be deriving ongoing value from the platform itself. They use it as a tool, not a workflow hub.
- **Collaboration (12.5%)** and **API Access (10.9%)** both have above-average churn — counterintuitive, as these are typically "power user" features. It may indicate that customers adopting these features are more demanding and leave when expectations aren't met.
- **AI Insights (10.0%)** churn is above average, suggesting the feature isn't yet delivering enough perceived value to retain customers who rely on it.

---
## 9. High-Risk Customer Profiles

**Business Question:** What does a typical churned customer look like?

By profiling churned customers across multiple attributes, we can build a description of the "typical" customer who leaves — which can inform proactive monitoring.

In [ ]:
# ── Most common attributes among churned customers ────────────────────────────
churned_df = master[master['is_churned'] == True].copy()

# Most common industry, size, plan, and top feature among churned customers
top_industry = churned_df['industry'].value_counts().index[0]
top_size     = churned_df['company_size'].value_counts().index[0]
top_plan     = churned_df['plan_name'].value_counts().index[0]
top_feature  = churned_df['top_feature'].value_counts().index[0]
top_reason   = churned_df['reason'].value_counts().index[0]

# Average usage metrics for churned customers
avg_sessions  = round(churned_df['avg_sessions'].mean(), 1)
avg_feat_use  = round(churned_df['avg_feature_usage'].mean(), 2)
avg_active_u  = round(churned_df['avg_active_users'].mean(), 1)

print('━' * 54)
print('   TYPICAL CHURNED CUSTOMER PROFILE')
print('━' * 54)
profile_data = [
    ['Most common industry',     top_industry],
    ['Most common company size', top_size],
    ['Most common plan',         top_plan],
    ['Most common top feature',  top_feature],
    ['Most common churn reason', top_reason],
    ['Avg sessions / month',     avg_sessions],
    ['Avg active users',         avg_active_u],
    ['Avg feature usage score',  avg_feat_use],
]
profile_df = pd.DataFrame(profile_data, columns=['Attribute', 'Typical Value'])
print(profile_df.to_string(index=False))
print('━' * 54)

In [ ]:
# ── Identify low-activity customers still active (at-risk signals) ────────────
# Churned customers averaged ~65 sessions/month
# Flag currently active customers with similar low session counts

low_threshold = 70   # sessions/month — near churned customer average

at_risk = master[
    (master['is_churned'] == False) &
    (master['avg_sessions'] < low_threshold)
][['customer_id', 'company_name', 'industry', 'company_size',
   'plan_name', 'avg_sessions', 'avg_feature_usage']].copy()

at_risk = at_risk.sort_values('avg_sessions').reset_index(drop=True)

print(f'Active customers with avg sessions < {low_threshold}/month (churn risk signal):')
print(f'Count: {len(at_risk)} customers\n')
print(at_risk.head(15).to_string(index=False))

# Industry breakdown of at-risk customers
print('\nAt-risk count by industry:')
print(at_risk['industry'].value_counts().to_string())

### 📌 Interpretation

**The typical churned NovaSuite customer:**
- Comes from the **EdTech industry**
- Is a **Small or Medium** sized company
- Is on the **Pro plan**
- Used **Reports or Dashboard** as their primary feature
- Left citing **Budget Cut** as the reason
- Averaged only **~65 sessions/month** and a **3.39/10 feature usage score** — well below active customers

**Active customers below the 70-session threshold** are exhibiting similar low-engagement behavior to customers who eventually churned. These customers represent the most actionable near-term risk pool — they are disengaged but haven't left yet.

---
## 10. Churn Findings Summary



---

### 🔍 Key Churn Findings

**Finding 1 — EdTech is a severe outlier**  
EdTech has a **15.0% churn rate** — more than double the 7.2% baseline. It represents 12% of customers but 25% of all churns. This pattern holds across all plan tiers (Basic, Pro, Business), suggesting an industry-wide product-fit issue rather than a single plan problem.

**Finding 2 — The Pro plan is the highest-churn tier**  
The Pro plan has a **10.17% churn rate** — the worst of any plan — and accounts for 18 of 36 total churns. Customers on Pro span industries and sizes, but the combination of Large companies on Pro (16.7% churn) and Small companies on Pro (10.96% churn) suggests both over- and under-provisioning relative to actual needs.

**Finding 3 — Budget and price are the dominant exit drivers**  
**58.3% of churns** are explained by financial reasons: Budget Cut (33.3%) or Too Expensive (25.0%). Only 19.4% of exits cite Lack of Features. This means the majority of churned customers were satisfied enough with the product but couldn't justify the cost.

**Finding 4 — Churned customers used the platform ~half as much**  
Churned customers averaged **65 sessions/month** vs 140 for active customers — a 53% gap. Login counts show the same 53% drop. This confirms that low usage is a statistically consistent precursor to churn, not random noise.

**Finding 5 — EdTech + Business plan is the worst single segment**  
With a **23.5% churn rate**, EdTech customers on the Business plan churn at 3× the overall rate. Of 17 such customers, 4 have already churned — an outsized loss concentrated in one identifiable cell.

**Finding 6 — Integrations users churn the least**  
Customers whose primary feature is Integrations have a **3.7% churn rate** — half the baseline. Dashboard users (4.1%) are similarly retained. In contrast, Data Export users churn at **19.2%** — the highest of any feature group — suggesting they extract value without deeply embedding the platform into their workflows.

**Finding 7 — Large companies on the Pro plan are disproportionately at risk**  
Large companies on Pro show a **16.7% churn rate** — the highest of any size-plan combination. These companies likely outgrew the Pro plan's capacity limits without upgrading to Business or Enterprise.

**Finding 8 — Enterprise-sized customers churn more than expected**  
Enterprise-sized companies have an **8.16% churn rate**, higher than Small (7.45%), Medium (7.27%), and Large (6.12%). Enterprise companies on the Business plan (13.6% churn) are especially notable — they are on a tier below Enterprise-level capacity.

**Finding 9 — A detectable pool of at-risk active customers exists today**  
Active customers currently averaging fewer than 70 sessions/month — the approximate usage level of churned customers — represent an identifiable at-risk group. This pool already shows the behavioral signature of customers who subsequently churned.

**Finding 10 — AI Insights adoption does not prevent churn**  
Despite AI Insights being a differentiating feature, customers primarily using it churn at **10.0%** — above the 7.2% baseline. This challenges the assumption that adopting AI features drives retention. Collaboration (12.5%) and API Access (10.9%) similarly over-churn despite being "power user" features.

